# TopLayer playground (single link)

Self-contained section: single-link run of **`TopLayer`** (the asynch 254/255
hillslope) from `models.py`, using `setup.py` (`precalculations_toplayer`,
`ArrayForcing`, `GLOBAL_PARAMS_TOPLAYER`).

**State vector** (dim = 3): `[s_p, s_t, s_s]`: ponded / top-layer /
sub-surface storage [m]. There is no channel routing: streamflow is the
hillslope outflow flux `q_out = (q_pl + q_sl)·c_2` [m³/s].

**Forcings:** rainfall [mm/hr], potential evap [mm/month]; time in **minutes**.

Variables in this section are prefixed `hlm_` so they don't clash with `q`,
`r`, `qs` above.

> `%autoreload` is enabled, so edits to `models.py` / `setup.py` take
> effect on the next cell run without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from setup import precalculations_toplayer, ArrayForcing, GLOBAL_PARAMS_TOPLAYER
from models import get_model, flux_timeseries

hlm_model = get_model("TopLayer")
print(hlm_model.name, "dim =", hlm_model.dim, hlm_model.state_names)
print(hlm_model.description)

# global-parameter index -> name (layout documented in setup.py)
HLM_GP_NAMES = {i: name for i, (name, unit) in enumerate(hlm_model.param_names)}
HLM_GP_NAMES

## Forcing
Synthetic hourly rain (10 days, three storms) and constant `Ep`. Replace `hlm_rain` with your own hourly series (e.g. a column of the AORC rainfall `r` above).

In [ ]:
n_hours = 240
hlm_rain = np.zeros(n_hours)                                    # [mm/hr]
hlm_rain[11:17]  = [5, 10, 17, 10, 8, 3]                        # storm 1
hlm_rain[60:65]  = [5, 15, 25, 10, 2]                           # storm 2 (biggest)
hlm_rain[110:114] = [2, 4, 4, 1]                                # small pulse
hlm_epot = 50.0                                                 # [mm/month]

hlm_forcing = ArrayForcing(hlm_rain, hlm_epot, dt=60.0)         # dt in MINUTES
hlm_t_end = hlm_forcing.t_end
print(f"{n_hours} h | total rain = {hlm_rain.sum():.0f} mm")

## Single run

In [ ]:
# --- global parameters (copy & edit to test alternative configurations) -----
hlm_gp = GLOBAL_PARAMS_TOPLAYER.copy()
hlm_gp[10] = 1.25e-3          # Ss_coef (same baseline as Hillslope_config_tester)

# --- single test link -------------------------------------------------------
hlm_L, hlm_A_h, hlm_A_i = 350.0, 133356.82501758, 50.0   # [m], [m^2], [km^2]
hlm_params = precalculations_toplayer(hlm_L, hlm_A_h, hlm_A_i, hlm_gp)

# initial state [s_p, s_t, s_s]
hlm_y0 = np.array([0.0, 0.05, 0.20])
assert len(hlm_y0) == hlm_model.dim

def hlm_run(gp, y0=hlm_y0, L=hlm_L, A_h=hlm_A_h, A_i=hlm_A_i, forcing=hlm_forcing):
    """Run the single link with global params `gp`; returns (solve_ivp result, params)."""
    p = precalculations_toplayer(L, A_h, A_i, gp)
    s = solve_ivp(
        hlm_model.dydt, (0.0, forcing.t_end), y0,
        args=(p, gp, forcing),
        method="LSODA", rtol=1e-6, atol=1e-9, max_step=30.0, dense_output=True,
    )
    return s, p

def hlm_outflow(sol, p, gp, forcing=hlm_forcing):
    """Hillslope outflow q_out [m^3/s] at the solution's time points."""
    return flux_timeseries(hlm_model, sol.t, sol.y, p, gp, forcing)["q_out"]

hlm_sol, hlm_params = hlm_run(hlm_gp)
print(hlm_sol.message, "| steps:", hlm_sol.t.size)

hlm_t_h = hlm_sol.t / 60.0
hlm_flux = flux_timeseries(hlm_model, hlm_sol.t, hlm_sol.y, hlm_params, hlm_gp, hlm_forcing)

In [ ]:
# --- plot the response ------------------------------------------------------
fig, (ax1, ax2, axf, axe) = plt.subplots(4, 1, figsize=(11, 13), sharex=True)

axr = ax1.twinx()
axr.bar(np.arange(hlm_forcing.n) + 0.5, hlm_forcing.rain, width=1.0,
        color="tab:blue", alpha=0.3, label="rain")
axr.set_ylabel("rain [mm/hr]")
axr.set_ylim(hlm_forcing.rain.max() * 3 if hlm_forcing.rain.max() > 0 else 1, 0)

c_2 = hlm_params[6]
ax1.plot(hlm_t_h, hlm_flux["q_out"], color="tab:red", lw=2.5, label="q_out (hillslope)")
ax1.plot(hlm_t_h, hlm_flux["q_sl"] * c_2, color="tab:orange", ls="--",
         label="q_sl·c_2 (sub-surface part)")
ax1.set_ylabel("discharge [m³/s]"); ax1.legend(loc="center right"); ax1.grid(alpha=0.3)
ax1.set_title(f"HLM '{hlm_model.name}' — single-link response")
ax1.set_zorder(axr.get_zorder() + 1); ax1.patch.set_visible(False)

for i, lab in ((0, "s_p (ponded)"), (1, "s_t (top layer)"), (2, "s_s (subsurface)")):
    ax2.plot(hlm_t_h, hlm_sol.y[i], lw=2, label=lab)
ax2.axhline(hlm_gp[8], color="tab:green", ls="--", lw=1, label="Ssmax")
ax2.axhline(hlm_gp[9], color="tab:green", ls=":",  lw=1, label="Ssmin")
ax2.set_ylabel("storage [m]"); ax2.legend(); ax2.grid(alpha=0.3)

for key in ("q_pl", "q_pt", "q_ts", "q_sl"):
    axf.plot(hlm_t_h, hlm_flux[key], lw=2, label=key)
axf.set_ylabel("flux [m/min]"); axf.set_yscale("symlog", linthresh=1e-9)
axf.legend(); axf.grid(alpha=0.3)

for key in ("e_p", "e_t", "e_s"):
    axe.plot(hlm_t_h, hlm_flux[key], lw=2, label=key)
axe.set_xlabel("time [h]"); axe.set_ylabel("evaporation [m/min]")
axe.legend(); axe.grid(alpha=0.3)

plt.tight_layout(); plt.show()
q_out = hlm_flux["q_out"]
print(f"peak q_out = {q_out.max():.4f} m³/s at t = {hlm_t_h[q_out.argmax()]:.1f} h")

## Hillslope water balance
`rain − (q_pl + q_sl + e_p + e_t + e_s) − Δ(s_p + s_t + s_s)` should be ~0 [mm over the hillslope].

In [ ]:
def hlm_water_balance(sol, params, gp, forcing, n_sub=60):
    """Hillslope mass balance [mm]; midpoint rule on the dense solution."""
    t0, t1 = sol.t[0], sol.t[-1]
    edges = np.linspace(t0, t1, int(np.ceil((t1 - t0) / 60.0 * n_sub)) + 1)
    tm, h = 0.5 * (edges[1:] + edges[:-1]), np.diff(edges)
    f = flux_timeseries(hlm_model, tm, sol.sol(tm), params, gp, forcing)
    I = {k: np.sum(v * h) * 1e3 for k, v in f.items()}          # m -> mm
    dS = (sol.y[0:3, -1] - sol.y[0:3, 0]).sum() * 1e3
    out = {"rain": I["rain_in"], "q_pl": I["q_pl"], "q_sl": I["q_sl"],
           "evap": I["e_p"] + I["e_t"] + I["e_s"], "dStorage": dS}
    out["residual"] = out["rain"] - out["q_pl"] - out["q_sl"] - out["evap"] - dS
    return out

pd.Series(hlm_water_balance(hlm_sol, hlm_params, hlm_gp, hlm_forcing), name="mm").round(4).to_frame()

# TopLayer parameter sensitivity (OAT)

Each global parameter is varied over its range while all others stay at the
baseline (`hlm_gp`). Outputs: **peak q_out**, **time to peak** and **total
volume** `∫q_out dt`. Elasticity = `%Δoutput / %Δparam`.

Swept: `v_h`, `k_I_factor`, `h_b`, `S_L`, `Ssmax`, `Ssmin`, `Ss_coef`
(log-spaced), `Ss_exp`. Combinations that break `S_L < h_b` or
`Ssmin < Ssmax` are skipped.

In [ ]:
hlm_base = hlm_gp.copy()

def _hlm_valid(gp):
    return gp[4] < gp[3] and gp[9] < gp[8]

def _hlm_metrics(gp):
    """(peak q_out [m³/s], time to peak [h], volume [m³]) for global params `gp`."""
    s, p = hlm_run(gp)
    if not s.success:
        return np.nan, np.nan, np.nan
    q = hlm_outflow(s, p, gp)
    return q.max(), s.t[q.argmax()] / 60.0, np.trapezoid(q, s.t * 60.0)

# global index -> sweep values
hlm_sweep_ranges = {
    0:  np.linspace(0.005, 0.1, 11),      # v_h
    2:  np.linspace(0.001, 0.1, 11),      # k_I_factor
    3:  np.linspace(0.2, 1.5, 11),        # h_b
    4:  np.linspace(0.02, 0.4, 11),       # S_L
    8:  np.linspace(0.05, 0.4, 11),       # Ssmax
    9:  np.linspace(0.0, 0.18, 11),       # Ssmin
    10: np.geomspace(1e-6, 1e-1, 11),     # Ss_coef
    11: np.linspace(5, 100, 11),          # Ss_exp
}

b_peak, b_tp, b_vol = _hlm_metrics(hlm_base)
rows = []
for idx, values in hlm_sweep_ranges.items():
    for v in values:
        gp = hlm_base.copy(); gp[idx] = v
        if not _hlm_valid(gp):
            continue
        peak, tp, vol = _hlm_metrics(gp)
        dpar = (v - hlm_base[idx]) / hlm_base[idx] if hlm_base[idx] != 0 else np.nan
        ok = np.isfinite(dpar) and abs(dpar) > 1e-9
        rows.append(dict(idx=idx, param=HLM_GP_NAMES[idx], value=v,
                         peak_q=peak, t_peak_h=tp, volume_m3=vol,
                         el_peak=((peak - b_peak) / b_peak) / dpar if ok else np.nan,
                         el_vol=((vol - b_vol) / b_vol) / dpar if ok else np.nan))
hlm_sens = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, idx in zip(axes.flat, hlm_sweep_ranges):
    d = hlm_sens[hlm_sens.idx == idx]
    ax.plot(d.value, d.peak_q, "o-", color="tab:blue")
    ax.axvline(hlm_base[idx], color="k", ls=":", lw=1)
    ax.set_title(f"[{idx}] {HLM_GP_NAMES[idx]}"); ax.set_xlabel(HLM_GP_NAMES[idx])
    ax.set_ylabel("peak q_out [m³/s]", color="tab:blue"); ax.grid(alpha=0.3)
    if idx == 10: ax.set_xscale("log")
    ax2 = ax.twinx()
    ax2.plot(d.value, d.volume_m3, "s--", color="tab:orange")
    ax2.set_ylabel("volume [m³]", color="tab:orange")
fig.suptitle("TopLayer OAT sensitivity — peak q_out (blue) and flow volume (orange)")
plt.tight_layout(); plt.show()

# mean |elasticity|: which parameters matter most?
hlm_sens.groupby("param")[["el_peak", "el_vol"]].apply(lambda d: d.abs().mean()).sort_values("el_peak", ascending=False)

## Interactive hydrograph comparison

Full hydrographs for a few values of each swept parameter (others at
baseline). Baseline in black; one color family per parameter, darker =
larger value. Use the legend to toggle traces (double-click a group to isolate it).

In [ ]:
import matplotlib as mpl
import plotly.graph_objects as go

# global index -> (sweep values, colormap)
hlm_hydro_sweep = {
    0:  (np.linspace(0.005, 0.1, 5),   "Greens"),    # v_h
    2:  (np.linspace(0.001, 0.1, 5),   "Purples"),   # k_I_factor
    3:  (np.linspace(0.2, 1.5, 5),     "Reds"),      # h_b
    4:  (np.linspace(0.02, 0.4, 5),    "YlOrBr"),    # S_L
    8:  (np.linspace(0.05, 0.4, 5),    "BuGn"),      # Ssmax
    9:  (np.linspace(0.0, 0.18, 5),    "PuRd"),      # Ssmin
    10: (np.geomspace(1e-6, 1e-1, 5),  "GnBu"),      # Ss_coef
    11: (np.linspace(5, 100, 5),       "YlGn"),      # Ss_exp
}

def _hlm_run_q(gp):
    s, p = hlm_run(gp)
    return s.t / 60.0, hlm_outflow(s, p, gp)

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(hlm_forcing.n) + 0.5, y=hlm_forcing.rain, name="rain",
                     marker_color="rgba(31,119,180,0.3)", yaxis="y2"))
t_b, q_b = _hlm_run_q(hlm_base)
fig.add_trace(go.Scatter(x=t_b, y=q_b, mode="lines", line=dict(color="black", width=3),
                         name="baseline"))

for idx, (values, cmap_name) in hlm_hydro_sweep.items():
    name, cmap = HLM_GP_NAMES[idx], mpl.colormaps[cmap_name]
    for k, v in enumerate(values):
        gp = hlm_base.copy(); gp[idx] = v
        if not _hlm_valid(gp):
            continue
        t_h, q = _hlm_run_q(gp)
        fig.add_trace(go.Scatter(
            x=t_h, y=q, mode="lines", legendgroup=name,
            line=dict(color=mpl.colors.to_hex(cmap(0.35 + 0.55 * k / (len(values) - 1))), width=1.5),
            name=f"{name} = {v:.3g}",
            hovertemplate=f"{name}={v:.3g}<br>t=%{{x:.1f}} h<br>q_out=%{{y:.4f}} m³/s<extra></extra>",
        ))

fig.update_layout(
    title="TopLayer hydrograph sensitivity",
    xaxis_title="time [h]", yaxis_title="q_out [m³/s]",
    yaxis2=dict(title="rain [mm/hr]", overlaying="y", side="right",
                range=[hlm_forcing.rain.max() * 3, 0]),
    height=600, hovermode="closest",
)
fig.show()

# Collie River basin 3 model Mod playground (single link)

Single-link run of **`CRB3`** from `models.py`, using `setup.py`
(`precalculations_crb3`, `GLOBAL_PARAMS_CRB3`). Uses the **same storms and
forcing** as the HLM section above (`hlm_forcing`) and the same test link
(`hlm_L`, `hlm_A_h`, `hlm_A_i`), so the two models can be compared directly.

**State vector** (dim = 4): `[Q, s_s, s_p, s_g]` — cumulative runoff [m], soil /
ponded / groundwater storage [m]. Streamflow is the flux
`q_out = (q_pL + q_gL + q_sL)·c_2` [m³/s].

**Units:** storages [m], fluxes [m/min], `A`, `A2` per minute.

Variables in this section are prefixed `crb_`.

In [ ]:
n_hours = 240
hlm_rain = np.zeros(n_hours)                                    # [mm/hr]
hlm_rain[11:17]  = [5, 10, 17, 10, 8, 3]                        # storm 1
hlm_rain[60:65]  = [5, 15, 25, 10, 2]                           # storm 2 (biggest)
hlm_rain[110:114] = [2, 4, 4, 1]                                # small pulse
hlm_epot = 50.0                                                 # [mm/month]

hlm_forcing = ArrayForcing(hlm_rain, hlm_epot, dt=60.0)         # dt in MINUTES
hlm_t_end = hlm_forcing.t_end
print(f"{n_hours} h | total rain = {hlm_rain.sum():.0f} mm")

In [ ]:
from setup import precalculations_crb3, GLOBAL_PARAMS_CRB3

crb_model = get_model("CRB3")
print(crb_model.name, "dim =", crb_model.dim, crb_model.state_names)
print(crb_model.description)

# global-parameter index -> name (layout documented in setup.py)
CRB_GP_NAMES = {0: "A", 1: "B", 2: "lambda", 3: "Ssmax", 4: "Sfc",
                5: "M", 6: "A2", 7: "B2", 8: "v_h"}

# --- global parameters (copy & edit to test alternative configurations) -----
crb_gp = GLOBAL_PARAMS_CRB3.copy()
crb_gp[0] = 1e-3        # A  [1/min]  (default 0.9 drains Ss in minutes -> very stiff)
crb_gp[7] = 1.0         # B2 [-]      (default 0.2 -> infinite dqgL/dSg at Sg=0, very stiff)

# initial state [Q, s_s, s_p, s_g]
crb_y0 = np.array([0.0, 0.05, 0.0, 0.001])
assert len(crb_y0) == crb_model.dim

def crb_run(gp, y0=crb_y0, L=hlm_L, A_h=hlm_A_h, A_i=hlm_A_i, forcing=hlm_forcing):
    """Run the single link with global params `gp`; returns (solve_ivp result, params)."""
    p = precalculations_crb3(L, A_h, A_i, gp)
    s = solve_ivp(
        crb_model.dydt, (0.0, forcing.t_end), y0,
        args=(p, gp, forcing),
        method="LSODA", rtol=1e-6, atol=1e-9, max_step=30.0, dense_output=True,
    )
    return s, p

crb_sol, crb_params = crb_run(crb_gp)
print(crb_sol.message, "| steps:", crb_sol.t.size)

crb_t_h = crb_sol.t / 60.0
crb_flux = flux_timeseries(crb_model, crb_sol.t, crb_sol.y, crb_params, crb_gp, hlm_forcing)

In [ ]:
# --- plot the response ------------------------------------------------------
fig, (ax1, ax2, axf, axe) = plt.subplots(4, 1, figsize=(11, 13), sharex=True)
crb_c2 = crb_params[6]

axr = ax1.twinx()
axr.bar(np.arange(hlm_forcing.n) + 0.5, hlm_forcing.rain, width=1.0,
        color="tab:blue", alpha=0.3, label="rain")
axr.set_ylabel("rain [mm/hr]")
axr.set_ylim(hlm_forcing.rain.max() * 3 if hlm_forcing.rain.max() > 0 else 1, 0)

ax1.plot(crb_t_h, crb_flux["q_out"], color="tab:red", lw=2.5, label="q_out (total)")
for key, ls in (("q_pL", "--"), ("q_sL", "-."), ("q_gL", ":")):
    ax1.plot(crb_t_h, crb_flux[key] * crb_c2, ls=ls, lw=1.5, label=f"{key}·c_2")
ax1.set_ylabel("discharge [m³/s]"); ax1.legend(loc="center right"); ax1.grid(alpha=0.3)
ax1.set_title(f"'{crb_model.name}' — single-link response")
ax1.set_zorder(axr.get_zorder() + 1); ax1.patch.set_visible(False)

for i, lab in ((1, "s_s (soil)"), (2, "s_p (ponded)"), (3, "s_g (groundwater)")):
    ax2.plot(crb_t_h, crb_sol.y[i], lw=2, label=lab)
ax2.axhline(crb_gp[3], color="tab:green", ls="--", lw=1, label="Ssmax")
ax2.axhline(crb_gp[4] * crb_gp[3], color="tab:green", ls=":", lw=1, label="Sfc·Ssmax")
ax2.set_ylabel("storage [m]"); ax2.legend(); ax2.grid(alpha=0.3)

for key in ("q_sp", "q_ss", "q_sg", "q_sL", "q_pL", "q_gL"):
    axf.plot(crb_t_h, crb_flux[key], lw=2, label=key)
axf.set_ylabel("flux [m/min]"); axf.set_yscale("symlog", linthresh=1e-9)
axf.legend(ncol=2); axf.grid(alpha=0.3)

for key in ("e_b", "e_v"):
    axe.plot(crb_t_h, crb_flux[key], lw=2, label=key)
axe.set_xlabel("time [h]"); axe.set_ylabel("evaporation [m/min]")
axe.legend(); axe.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"peak q_out = {crb_flux['q_out'].max():.4f} m³/s at t = {crb_t_h[crb_flux['q_out'].argmax()]:.1f} h"
      f" | cumulative Q = {crb_sol.y[0, -1] * 1e3:.1f} mm")

## Hillslope water balance
`rain − (q_pL + q_sL + q_gL + e_b + e_v) − Δ(s_s + s_p + s_g)` should be ~0 [mm over the hillslope].
`ΔQ − (q_pL + q_sL + q_gL)` checks the cumulative-runoff state against the integrated outflow.

In [ ]:
def crb_water_balance(sol, params, gp, forcing, n_sub=60):
    """Hillslope mass balance [mm]; midpoint rule on the dense solution."""
    t0, t1 = sol.t[0], sol.t[-1]
    edges = np.linspace(t0, t1, int(np.ceil((t1 - t0) / 60.0 * n_sub)) + 1)
    tm, h = 0.5 * (edges[1:] + edges[:-1]), np.diff(edges)
    f = flux_timeseries(crb_model, tm, sol.sol(tm), params, gp, forcing)
    I = {k: np.sum(v * h) * 1e3 for k, v in f.items()}          # m -> mm
    dS = (sol.y[1:4, -1] - sol.y[1:4, 0]).sum() * 1e3
    out = {"rain": I["rain_in"], "q_pL": I["q_pL"], "q_sL": I["q_sL"], "q_gL": I["q_gL"],
           "evap": I["e_b"] + I["e_v"], "dStorage": dS}
    runoff = out["q_pL"] + out["q_sL"] + out["q_gL"]
    out["residual"] = out["rain"] - runoff - out["evap"] - dS
    out["dQ - runoff"] = (sol.y[0, -1] - sol.y[0, 0]) * 1e3 - runoff
    return out

pd.Series(crb_water_balance(crb_sol, crb_params, crb_gp, hlm_forcing), name="mm").round(4).to_frame()

# CRB3 parameter sensitivity (OAT)

Each global parameter is varied over its range while all others stay at the
baseline (`crb_gp`). Outputs: **peak q_out**, **time to peak** and **total
volume** `∫q_out dt`. Elasticity = `%Δoutput / %Δparam`.

Swept: all 9 parameters; `A` and `A2` log-spaced. Runs that fail to
integrate are reported as NaN.

In [ ]:
crb_base = crb_gp.copy()

def _crb_q_out(s, p, gp):
    """Streamflow q_out [m³/s] at the solver time points."""
    return flux_timeseries(crb_model, s.t, s.y, p, gp, hlm_forcing)["q_out"]

def _crb_metrics(gp):
    """(peak q_out [m³/s], time to peak [h], volume [m³]) for global params `gp`."""
    s, p = crb_run(gp)
    if not s.success:
        return np.nan, np.nan, np.nan
    q = _crb_q_out(s, p, gp)
    return q.max(), s.t[q.argmax()] / 60.0, np.trapezoid(q, s.t * 60.0)

# global index -> sweep values
crb_sweep_ranges = {
    0: np.geomspace(1e-5, 1e-1, 11),      # A
    1: np.linspace(0.5, 3.0, 11),         # B
    2: np.linspace(0.0, 1.0, 11),         # lambda
    3: np.linspace(0.05, 0.5, 11),        # Ssmax
    4: np.linspace(0.1, 0.9, 11),         # Sfc
    5: np.linspace(0.0, 1.0, 11),         # M
    6: np.geomspace(1e-5, 1e-1, 11),      # A2
    7: np.linspace(0.5, 3.0, 11),         # B2
    8: np.linspace(0.005, 0.1, 11),       # v_h
}
CRB_LOG_PARAMS = {0, 6}

b_peak, b_tp, b_vol = _crb_metrics(crb_base)
rows = []
for idx, values in crb_sweep_ranges.items():
    for v in values:
        gp = crb_base.copy(); gp[idx] = v
        peak, tp, vol = _crb_metrics(gp)
        dpar = (v - crb_base[idx]) / crb_base[idx] if crb_base[idx] != 0 else np.nan
        ok = np.isfinite(dpar) and abs(dpar) > 1e-9
        rows.append(dict(idx=idx, param=CRB_GP_NAMES[idx], value=v,
                         peak_q=peak, t_peak_h=tp, volume_m3=vol,
                         el_peak=((peak - b_peak) / b_peak) / dpar if ok else np.nan,
                         el_vol=((vol - b_vol) / b_vol) / dpar if ok else np.nan))
crb_sens = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, idx in zip(axes.flat, crb_sweep_ranges):
    d = crb_sens[crb_sens.idx == idx]
    ax.plot(d.value, d.peak_q, "o-", color="tab:blue")
    ax.axvline(crb_base[idx], color="k", ls=":", lw=1)
    ax.set_title(f"[{idx}] {CRB_GP_NAMES[idx]}"); ax.set_xlabel(CRB_GP_NAMES[idx])
    ax.set_ylabel("peak q_out [m³/s]", color="tab:blue"); ax.grid(alpha=0.3)
    if idx in CRB_LOG_PARAMS: ax.set_xscale("log")
    ax2 = ax.twinx()
    ax2.plot(d.value, d.volume_m3, "s--", color="tab:orange")
    ax2.set_ylabel("volume [m³]", color="tab:orange")
for ax in axes.flat[len(crb_sweep_ranges):]:
    ax.set_visible(False)
fig.suptitle("CRB3 OAT sensitivity — peak q_out (blue) and flow volume (orange)")
plt.tight_layout(); plt.show()

# mean |elasticity|: which parameters matter most?
crb_sens.groupby("param")[["el_peak", "el_vol"]].apply(lambda d: d.abs().mean()).sort_values("el_peak", ascending=False)

## Interactive hydrograph comparison

Full `q_out` hydrographs for a few values of each swept parameter (others at
baseline). Baseline in black; one color family per parameter, darker =
larger value. Use the legend to toggle traces (double-click a group to isolate it).

In [ ]:
import matplotlib as mpl
import plotly.graph_objects as go

# global index -> (sweep values, colormap)
crb_hydro_sweep = {
    0: (np.geomspace(1e-5, 1e-1, 5),  "Blues"),     # A
    1: (np.linspace(0.5, 3.0, 5),     "Oranges"),   # B
    2: (np.linspace(0.0, 1.0, 5),     "Greens"),    # lambda
    3: (np.linspace(0.05, 0.5, 5),    "Purples"),   # Ssmax
    4: (np.linspace(0.1, 0.9, 5),     "Reds"),      # Sfc
    5: (np.linspace(0.0, 1.0, 5),     "YlOrBr"),    # M
    6: (np.geomspace(1e-5, 1e-1, 5),  "BuGn"),      # A2
    7: (np.linspace(0.5, 3.0, 5),     "PuRd"),      # B2
    8: (np.linspace(0.005, 0.1, 5),   "GnBu"),      # v_h
}

def _crb_run_q(gp):
    s, p = crb_run(gp)
    return s.t / 60.0, _crb_q_out(s, p, gp)

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(hlm_forcing.n) + 0.5, y=hlm_forcing.rain, name="rain",
                     marker_color="rgba(31,119,180,0.3)", yaxis="y2"))
t_b, q_b = _crb_run_q(crb_base)
fig.add_trace(go.Scatter(x=t_b, y=q_b, mode="lines", line=dict(color="black", width=3),
                         name="baseline"))

for idx, (values, cmap_name) in crb_hydro_sweep.items():
    name, cmap = CRB_GP_NAMES[idx], mpl.colormaps[cmap_name]
    for k, v in enumerate(values):
        gp = crb_base.copy(); gp[idx] = v
        t_h, q = _crb_run_q(gp)
        fig.add_trace(go.Scatter(
            x=t_h, y=q, mode="lines", legendgroup=name,
            line=dict(color=mpl.colors.to_hex(cmap(0.35 + 0.55 * k / (len(values) - 1))), width=1.5),
            name=f"{name} = {v:.3g}",
            hovertemplate=f"{name}={v:.3g}<br>t=%{{x:.1f}} h<br>q_out=%{{y:.4f}} m³/s<extra></extra>",
        ))

fig.update_layout(
    title="CRB3 hydrograph sensitivity",
    xaxis_title="time [h]", yaxis_title="q_out [m³/s]",
    yaxis2=dict(title="rain [mm/hr]", overlaying="y", side="right",
                range=[hlm_forcing.rain.max() * 3, 0]),
    height=600, hovermode="closest",
)
fig.show()